In [2]:
import datetime as dt

import great_tables as gt
import matplotlib.pyplot as plt
import polars as pl
import sf_quant.data as sfd
import sf_quant.performance as sfp
import sf_quant.research as sfr
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score
import numpy as np

In [3]:
production = pl.scan_parquet("alphas.parquet").collect()

production = production.pivot(on='signal_name', index=['date','barrid'], values='alpha')

production

date,barrid,momentum,reversal,beta,barra_reversal,barra_momentum,ivol
date,str,f64,f64,f64,f64,f64,f64
2012-06-28,"""GERFUS1""",0.004519,0.006309,0.020492,0.012333,-0.000227,0.005726
2012-06-29,"""GERFUS1""",0.004962,0.012323,0.019347,0.003672,-0.000562,0.005809
2012-07-02,"""GERFUS1""",0.005142,0.012752,0.019179,0.000106,-0.00039,0.00611
2012-07-03,"""GERFUS1""",0.005912,0.014468,0.018685,0.006334,-0.001531,0.006144
2012-07-05,"""GERFUS1""",0.004502,0.008738,0.018775,-0.011728,-0.002047,0.006012
…,…,…,…,…,…,…,…
2026-01-26,"""USBRWZ1""",null,null,0.006386,-0.035652,null,0.002044
2026-01-27,"""USBRWZ1""",null,null,0.006625,-0.008012,null,0.002341
2026-01-28,"""USBRWZ1""",null,null,0.006792,0.001273,null,0.002531


In [4]:
# Parameters
start = dt.date(1996, 1, 1)
end = dt.date(2024, 12, 31)
price_filter = 5
signal_name = "ols_ew"
IC = 0.05
gamma = 35


# Get data
data = sfd.load_assets(
    start=start,
    end=end,
    columns=[
        "date",
        "barrid",
        "ticker",
        "price",
        "return",
        "specific_return",
        "specific_risk",
        "predicted_beta",
    ],
    in_universe=True,
).with_columns(
    pl.col("return").truediv(100),
    pl.col("specific_return").truediv(100),
    pl.col("specific_risk").truediv(100),
)

exposures = sfd.load_exposures(
    start=start,
    end=end,
    in_universe=True,
    columns=[
        "date",
        "barrid",
        "USSLOWL_EARNQLTY",
        "USSLOWL_VALUE",
        "USSLOWL_PROFIT",
    ],
)

quality_cols = [
    "USSLOWL_PROFIT",
    "USSLOWL_EARNQLTY",
    "USSLOWL_MGMTQLTY",
    "USSLOWL_LEVERAGE",
    "USSLOWL_GROWTH",
]

signal_weights = {
    "USSLOWL_PROFIT": 1.2695,
    "USSLOWL_EARNQLTY": 1.0174,
    "USSLOWL_MGMTQLTY": 1.3337,
    "USSLOWL_LEVERAGE": -1.3200,
    "USSLOWL_GROWTH": -1.3006
}

factors = sfd.load_exposures(
    start=start, end=end, in_universe=True, columns=["date", "barrid"] + quality_cols
).with_columns(
    [
        (
            (pl.col(f) - pl.col(f).mean().over("date")) / pl.col(f).std().over("date")
        ).alias(f)
        for f in quality_cols
    ]
)

data = data.join(exposures, on=["date", "barrid"], how="left")

data = data.join(factors, on=["date", "barrid"], how="inner")

# static QMJ signal
signals = (
    data
    .sort("barrid", "date")
    .with_columns(
        sum(
            pl.col(factor) * signal_weights[factor]
            for factor in quality_cols
        )
        .shift(2)
        .over('barrid')
        .alias(signal_name)
    )
)

# Filter universe
filtered = signals.filter(
    pl.col("price").shift(1).over("barrid").gt(price_filter),
    pl.col(signal_name).is_not_null(),
    pl.col(signal_name).is_not_nan(),
)

# Compute scores
scores = filtered.select(
    "date",
    "barrid",
    "predicted_beta",
    "specific_risk",
    pl.col(signal_name)
    .sub(pl.col(signal_name).mean())
    .truediv(pl.col(signal_name).std())
    .over("date")
    .alias("score"),
)

# Compute alphas
alphas = (
    scores.with_columns(pl.col("score").mul(IC).mul("specific_risk").alias("alpha"))
    .select("date", "barrid", "alpha", "predicted_beta")
    .sort("date", "barrid")
)

In [5]:
alphas

date,barrid,alpha,predicted_beta
date,str,f64,f64
1996-01-04,"""USAA191""",0.004104,0.823868
1996-01-04,"""USAA1Y1""",0.025892,1.972125
1996-01-04,"""USAA2L1""",0.004025,0.385132
1996-01-04,"""USAA311""",-0.007785,0.861978
1996-01-04,"""USAA3I1""",-0.021614,1.186341
…,…,…,…
2024-12-31,"""USBQFF1""",-0.015339,0.850589
2024-12-31,"""USBQGD1""",-0.030158,1.255632
2024-12-31,"""USBQLB1""",0.004695,1.312122


In [6]:
merged = (
    production
    .join(alphas, on=['date','barrid'], how='left')
    .sort(['date','barrid'])
    .drop(['momentum','barra_momentum','predicted_beta','reversal'])
    .drop_nulls()
    .rename({'alpha': 'quality'})
)

merged

date,barrid,beta,barra_reversal,ivol,quality
date,str,f64,f64,f64,f64
1996-01-04,"""USAA191""",0.004793,0.011349,0.006503,0.004104
1996-01-04,"""USAA311""",0.004577,0.014502,0.003234,-0.007785
1996-01-04,"""USAA4D1""",-0.027764,0.014614,-0.019031,-0.030049
1996-01-04,"""USAA4H1""",0.006692,0.003591,0.008284,0.004889
1996-01-04,"""USAA6F1""",-0.017497,-0.045883,-0.043253,0.036361
…,…,…,…,…,…
2024-12-31,"""USBPZP1""",-0.070403,0.030859,-0.141889,0.026735
2024-12-31,"""USBQBP1""",0.010345,-0.000292,0.008281,0.001502
2024-12-31,"""USBQDU1""",0.00823,-0.014603,-0.003639,-0.012049


In [7]:
# pairwise correlations

merged.select([
    pl.corr('beta','quality'),
    pl.corr('barra_reversal','quality'),
    pl.corr('ivol','quality')
])

beta,barra_reversal,ivol
f64,f64,f64
0.385586,0.004632,0.411188


In [8]:
# joint correlation/R-squared

X = merged[['beta','barra_reversal','ivol']].to_numpy()
y = merged['quality'].to_numpy()

model = LinearRegression().fit(X,y)
print('R2:', model.score(X,y))

R2: 0.203192509542638


In [9]:
# gather residuals

residuals = y - model.predict(X)

residuals

array([ 0.00266414, -0.00834984, -0.01718804, ..., -0.01183285,
       -0.01792135,  0.01183107], shape=(7860249,))

In [10]:
merged = merged.with_columns(pl.Series('residuals', residuals))

merged

date,barrid,beta,barra_reversal,ivol,quality,residuals
date,str,f64,f64,f64,f64,f64
1996-01-04,"""USAA191""",0.004793,0.011349,0.006503,0.004104,0.002664
1996-01-04,"""USAA311""",0.004577,0.014502,0.003234,-0.007785,-0.00835
1996-01-04,"""USAA4D1""",-0.027764,0.014614,-0.019031,-0.030049,-0.017188
1996-01-04,"""USAA4H1""",0.006692,0.003591,0.008284,0.004889,0.002528
1996-01-04,"""USAA6F1""",-0.017497,-0.045883,-0.043253,0.036361,0.05268
…,…,…,…,…,…,…
2024-12-31,"""USBPZP1""",-0.070403,0.030859,-0.141889,0.026735,0.080707
2024-12-31,"""USBQBP1""",0.010345,-0.000292,0.008281,0.001502,-0.001752
2024-12-31,"""USBQDU1""",0.00823,-0.014603,-0.003639,-0.012049,-0.011833


In [11]:
alphas.join(merged, on=['date','barrid'], how='left').fill_null(0).select(['date','barrid','residuals','predicted_beta']).rename({'residuals':'alpha'})

date,barrid,alpha,predicted_beta
date,str,f64,f64
1996-01-04,"""USAA191""",0.002664,0.823868
1996-01-04,"""USAA1Y1""",0.0,1.972125
1996-01-04,"""USAA2L1""",0.0,0.385132
1996-01-04,"""USAA311""",-0.00835,0.861978
1996-01-04,"""USAA3I1""",0.0,1.186341
…,…,…,…
2024-12-31,"""USBQFF1""",-0.017921,0.850589
2024-12-31,"""USBQGD1""",0.0,1.255632
2024-12-31,"""USBQLB1""",0.011831,1.312122
